In [1]:
from datasets import load_from_disk, load_dataset
import pandas as pd
import json

/opt/miniconda3/envs/fm_312_hf_latest/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_from_disk("MULAN_with_questions")
dataset

Dataset({
    features: ['query', 'answer', 'id', 'relation', 'date', 'type', 'extracted_subject', 'selected_answer', 'prompt', 'messages', 'context', 'selected_fact', 'question'],
    num_rows: 49049
})

In [3]:
df = dataset.to_pandas()
df.head()

,query,answer,id,relation,date,type,extracted_subject,selected_answer,prompt,messages,context,selected_fact,question
0,Barack Obama works in the field of _X_.,"[{'wikidata_id': 'Q25447176', 'name': 'civil r...",Q76_P101_0,P101,2021,immutable_n,"{'subject': 'Barack Obama', 'template': '[X] w...","{'name': 'constitutional law', 'wikidata_id': ...",\nCreate a small paragraph (3-4 sentences) tha...,[{'content': 'You are an expert Wikipedia edit...,Barack Obama is an American politician and att...,Barack Obama works in the field of constitutio...,What field does Barack Obama work in?
1,Jesus works in the field of _X_.,"[{'wikidata_id': 'Q203605', 'name': 'carpentry...",Q302_P101_0,P101,2021,immutable_n,"{'subject': 'Jesus', 'template': '[X] works in...","{'name': 'carpentry', 'wikidata_id': 'Q203605'}",\nCreate a small paragraph (3-4 sentences) tha...,[{'content': 'You are an expert Wikipedia edit...,Jesus is believed to have been born in Bethleh...,Jesus works in the field of carpentry,What field does Jesus work in?
2,William Shakespeare works in the field of _X_.,"[{'wikidata_id': 'Q8253972', 'name': 'fiction'...",Q692_P101_0,P101,2021,immutable_n,"{'subject': 'William Shakespeare', 'template':...","{'name': 'fiction', 'wikidata_id': 'Q8253972'}",\nCreate a small paragraph (3-4 sentences) tha...,[{'content': 'You are an expert Wikipedia edit...,William Shakespeare is widely regarded as one ...,William Shakespeare works in the field of fiction,What field does William Shakespeare work in?
3,Adolf Hitler works in the field of _X_.,"[{'wikidata_id': 'Q7310', 'name': 'Nazism'}, {...",Q352_P101_0,P101,2021,immutable_n,"{'subject': 'Adolf Hitler', 'template': '[X] w...","{'name': 'Nazi propaganda', 'wikidata_id': 'Q7...",\nCreate a small paragraph (3-4 sentences) tha...,[{'content': 'You are an expert Wikipedia edit...,Adolf Hitler was a key figure in the developme...,Adolf Hitler works in the field of Nazi propag...,What field does Adolf Hitler work in?
4,Donald Trump works in the field of _X_.,"[{'wikidata_id': 'Q10494269', 'name': 'real pr...",Q22686_P101_0,P101,2021,immutable_n,"{'subject': 'Donald Trump', 'template': '[X] w...","{'name': 'government', 'wikidata_id': 'Q7188'}",\nCreate a small paragraph (3-4 sentences) tha...,[{'content': 'You are an expert Wikipedia edit...,"Donald Trump is an American businessman, telev...",Donald Trump works in the field of government,What field does Donald Trump work in?


In [4]:
with open("aliases.json", "r") as f:
    aliases = json.load(f)
len(aliases)

25179

In [5]:
def get_subject(example):
    subj = example["extracted_subject"]["subject"]
    example["subj"] = subj
    return example

def get_obj(example):
    obj = example["selected_answer"]["name"]
    example["obj"] = obj
    return example

def get_subj_id(example):
    subj_id = example["id"].split("_")[0]
    example["subj_id"] = subj_id
    return example

def get_obj_id(example):
    obj_id = example["selected_answer"]["wikidata_id"]
    example["obj_id"] = obj_id
    return example

def get_possible_answers(example):
    possible_answers = []
    for ans in example["answer"]:
        possible_answers.append(ans["name"])
        if ans["wikidata_id"] in aliases:
            possible_answers.extend(aliases[ans["wikidata_id"]])
    example["possible_answers"] = possible_answers
    return example

dataset = dataset.map(get_subject)
dataset = dataset.map(get_obj)
dataset = dataset.map(get_subj_id)
dataset = dataset.map(get_obj_id)
dataset = dataset.map(get_possible_answers)

Map: 100%|██████████| 49049/49049 [00:05<00:00, 8873.27 examples/s] 


In [6]:
dataset[0]

{'query': 'Barack Obama works in the field of _X_.',
 'answer': [{'wikidata_id': 'Q25447176', 'name': 'civil rights'},
  {'wikidata_id': 'Q11206', 'name': 'constitutional law'}],
 'id': 'Q76_P101_0',
 'relation': 'P101',
 'date': 2021,
 'type': 'immutable_n',
 'extracted_subject': {'subject': 'Barack Obama',
  'template': '[X] works in the field of [Y].'},
 'selected_answer': {'name': 'constitutional law', 'wikidata_id': 'Q11206'},
 'prompt': '\nCreate a small paragraph (3-4 sentences) that supports a given natural language fact. The generated paragraph should feel like it was copied from a larger Wikipedia article about the Subject.\n\nFollow these rules strictly:\n1. CRITICAL: The text MUST contain the exact Object string provided, without modification, suffixes, or alternative forms.\n2. CRITICAL: Do NOT include any plurals, derivatives, or related forms of the Object.\n3. The text must be about the provided Subject and support the information in the Fact Sentence.\n4. The text must